# TruthLens: Exploratory Data Analysis (EDA)
### AI-Powered Fake News Detection for Social Good

This notebook performs thorough exploratory data analysis on the news corpus:
- Class balance analysis (Real vs Fake distribution)
- Document length and character distributions
- Uppercase and exclamation mark ratios across classes
- Most prominent n-grams per class
- Anti-leakage verification (ensuring publisher artifacts are purged)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
from pathlib import Path

# Set style
sns.set_theme(style="whitegrid", font="sans-serif")
DATA_PATH = Path("../data/processed/news_dataset.csv")
df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 1. Class Distribution Analysis

In [ ]:
counts = df['label_name'].value_counts()
plt.figure(figsize=(6, 4), dpi=150)
sns.barplot(x=counts.index, y=counts.values, palette=['#287c70', '#e46c50'])
plt.title("Class Distribution: Verified Real vs Fabricated Fake News", fontsize=12, fontweight='bold')
plt.ylabel("Count")
for i, v in enumerate(counts.values):
    plt.text(i, v + 15, str(v), ha='center', fontweight='bold')
plt.show()

## 2. Text Length & Word Count Distributions

In [ ]:
df['word_count'] = df['text'].apply(lambda t: len(str(t).split()))
df['caps_ratio'] = df['text'].apply(lambda t: sum(1 for c in str(t) if c.isupper()) / max(1, sum(1 for c in str(t) if c.isalpha())))
df['exclamation_count'] = df['text'].apply(lambda t: str(t).count('!'))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=150)
sns.boxplot(x='label_name', y='word_count', data=df, ax=axes[0], palette=['#287c70', '#e46c50'])
axes[0].set_title("Word Count Distribution by Class", fontweight='bold')

sns.boxplot(x='label_name', y='caps_ratio', data=df, ax=axes[1], palette=['#287c70', '#e46c50'])
axes[1].set_title("Capitalization Ratio by Class", fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Top Salient N-Grams per Class

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def get_top_ngrams(corpus, n=2, top_k=10):
    vec = CountVectorizer(ngram_range=(n, n), stop_words='english').fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
    return words_freq[:top_k]

top_real_bigrams = get_top_ngrams(df[df['label']==0]['text'], n=2, top_k=8)
top_fake_bigrams = get_top_ngrams(df[df['label']==1]['text'], n=2, top_k=8)

print("Top Bigrams in Verified Real News:", top_real_bigrams)
print("Top Bigrams in Disinformation / Fake News:", top_fake_bigrams)